In [3]:
import os
import shutil
import deeplabcut
import ruamel.yaml

# --- PATHS ---
VIDEO_PATH = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_t7w1_redlinear_GX010260.MP4'
OLD_PROJ = '/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27'
BACKUP_DIR = '/mnt/Data/Projects/cloud_deployment/frame_backup'
WORKING_DIR = '/mnt/Data/Projects/cloud_deployment/Scripts'

# 1. Backup Frames
video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
frame_source = os.path.join(OLD_PROJ, 'labeled-data', video_name)

if os.path.exists(frame_source):
    if os.path.exists(BACKUP_DIR): shutil.rmtree(BACKUP_DIR)
    shutil.copytree(frame_source, BACKUP_DIR)
    print(f"✅ Frames backed up to {BACKUP_DIR}")

# 2. Delete the Poisoned Project
if os.path.exists(OLD_PROJ):
    shutil.rmtree(OLD_PROJ)
    print("🗑️ Broken project deleted.")

# 3. Fresh Initialization (v7.6 Standard)
config_path = deeplabcut.create_new_project(
    'Cloud_Deployment_SER', 'Dev', [VIDEO_PATH], 
    working_directory=WORKING_DIR, copy_videos=False
)

# 4. Inject 4-Point Skeleton (No Pectorals)
ryaml = ruamel.yaml.YAML()
ryaml.preserve_quotes = True
with open(config_path, 'r') as f:
    cfg = ryaml.load(f)

cfg['bodyparts'] = ['Snout', 'DorsalFin', 'TailBase', 'TailTip']
cfg['skeleton'] = [['Snout', 'DorsalFin'], ['DorsalFin', 'TailBase'], ['TailBase', 'TailTip']]
cfg['numframes2pick'] = 150

with open(config_path, 'w') as f:
    ryaml.dump(cfg, f)

print(f"✅ New project initialized at: {config_path}")

✅ Frames backed up to /mnt/Data/Projects/cloud_deployment/frame_backup
🗑️ Broken project deleted.
Created "/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/videos"
Created "/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/labeled-data"
Created "/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/training-datasets"
Created "/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/dlc-models"
Attempting to create a symbolic link of the video ...
Created the symlink of /mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_t7w1_redlinear_GX010260.MP4 to /mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/videos/20260318SER_t7w1_redlinear_GX010260.MP4
/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/videos/20260318SER_t7w1_redlinear_GX010260.MP4
Generated "/mnt/Data/Projects/cloud_deployment/Scripts/Cloud

In [4]:
import pandas as pd
import json
import numpy as np

# --- SETTINGS ---
ls_csv = os.path.expanduser('~/Downloads/project-9-at-2026-03-28-00-48-6f62bcab.csv')
new_frame_dest = os.path.join(os.path.dirname(config_path), 'labeled-data', video_name)
os.makedirs(new_frame_dest, exist_ok=True)

# 1. Restore Frames
for img in os.listdir(BACKUP_DIR):
    if img.endswith('.png'):
        shutil.copy(os.path.join(BACKUP_DIR, img), new_frame_dest)
print("🖼️ Frames restored to new project.")

# 2. Convert CSV (The 'Michael' split logic)
df_ls = pd.read_csv(ls_csv)
data_rows = []

for _, row in df_ls.iterrows():
    json_col = 'label' if 'label' in df_ls.columns else 'annotation'
    # Extract filename from LS path
    fname = os.path.basename(row['image']).split('-')[-1] if '-' in row['image'] else os.path.basename(row['image'])
    rel_path = f"labeled-data/{video_name}/{fname}"
    
    frame_data = {'image': rel_path}
    for bp in cfg['bodyparts']:
        frame_data[f'{bp}_x'], frame_data[f'{bp}_y'] = np.nan, np.nan
        
    try:
        labels = json.loads(row[json_col])
        if isinstance(labels, list) and 'result' in labels[0]: labels = labels[0]['result']
        for l in labels:
            v = l.get('value', {})
            bp = v.get('keypointlabels', [None])[0]
            if bp in cfg['bodyparts']:
                frame_data[f'{bp}_x'] = (v['x'] * v.get('original_width', 3840)) / 100.0
                frame_data[f'{bp}_y'] = (v['y'] * v.get('original_height', 2160)) / 100.0
    except: pass
    data_rows.append(frame_data)

# 3. Build Multi-Index H5
df_final = pd.DataFrame(data_rows).set_index('image')
multi_cols = pd.MultiIndex.from_product([['Dev'], cfg['bodyparts'], ['x', 'y']], names=['scorer', 'bodyparts', 'coords'])
df_final.columns = multi_cols
df_final.to_hdf(os.path.join(new_frame_dest, 'CollectedData_Dev.h5'), key='df_with_missing', mode='w')
df_final.to_csv(os.path.join(new_frame_dest, 'CollectedData_Dev.csv'))
print("✅ Labels imported successfully.")

🖼️ Frames restored to new project.
✅ Labels imported successfully.


In [9]:
import glob
import yaml
import deeplabcut
import os

# --- 1. SETTINGS ---
project_path = '/mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27'
config_path = os.path.join(project_path, 'config.yaml')
video_path = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_t7w1_redlinear_GX010260.MP4'

# --- 2. HARD-TYPE REPAIR ---
# We explicitly define which keys MUST be numeric to prevent the pow() error
INT_KEYS = ['batch_size', 'decay_steps', 'display_iters', 'maxiters', 'save_iters', 'crop_pad']
FLOAT_KEYS = ['global_scale', 'alpha_r', 'apply_prob', 'cropratio', 'weight_decay', 'scale_jitter_lo', 'scale_jitter_up']
BOOL_KEYS = ['allow_growth', 'deterministic', 'weigh_negatives', 'histeq', 'clahe']

pose_configs = glob.glob(os.path.join(project_path, 'dlc-models', '*', '*', '*', 'pose_cfg.yaml'), recursive=True)

def repair_yaml_types(file_path):
    # Load with standard yaml to strip the PosixPath tags immediately
    with open(file_path, 'r') as f:
        data = yaml.safe_load(f)
    
    def fix(obj):
        if isinstance(obj, dict):
            new_obj = {}
            for k, v in obj.items():
                # Force types based on key name
                if k in INT_KEYS and v is not None:
                    new_obj[k] = int(float(v))
                elif k in FLOAT_KEYS and v is not None:
                    new_obj[k] = float(v)
                elif k in BOOL_KEYS and v is not None:
                    new_obj[k] = str(v).lower() == 'true'
                elif isinstance(v, (dict, list)):
                    new_obj[k] = fix(v)
                elif hasattr(v, '__str__'): # Handles remaining path objects
                    new_obj[k] = str(v)
                else:
                    new_obj[k] = v
            return new_obj
        elif isinstance(obj, list):
            return [fix(v) for v in obj]
        return obj

    repaired_data = fix(data)
    
    # Save with standard yaml to keep it clean for the TF trainer
    with open(file_path, 'w') as f:
        yaml.dump(repaired_data, f, default_flow_style=False)

for p in pose_configs:
    repair_yaml_types(p)
    print(f"✅ Type-repaired: {p}")

# --- 3. START TRAINING & ANALYSIS ---
print("\n🔥 Starting training run. It's safe to let it chug now.")
try:
    # We use your notebook's preferred allow_growth=True for the 8GB VRAM limit
    deeplabcut.train_network(config_path, maxiters=80000, displayiters=1000, allow_growth=True)
    
    # Analysis steps (The "Morning Report")
    crop = [0, 3840, 700, 2160]
    deeplabcut.evaluate_network(config_path, plotting=True, show_errors=True)
    deeplabcut.analyze_videos(config_path, [video_path], save_as_csv=True, batchsize=1, cropping=crop)
    deeplabcut.create_labeled_video(config_path, [video_path], draw_skeleton=True)
    
    print("\n🎉 Morning report complete.")
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")

Config:
{'all_joints': [['0'], ['1'], ['2'], ['3']],
 'all_joints_names': ['Snout', 'DorsalFin', 'TailBase', 'TailTip'],
 'alpha_r': 0.02,
 'apply_prob': 0.5,
 'batch_size': 1,
 'contrast': {'clahe': True,
              'claheratio': '0.1',
              'histeq': True,
              'histeqratio': '0.1'},
 'convolution': {'edge': 'False',
                 'emboss': {'alpha': ['0.0', '1.0'],
                            'strength': ['0.5', '1.5']},
                 'embossratio': '0.1',
                 'sharpen': 'False',
                 'sharpenratio': '0.3'},
 'crop_pad': 0,
 'cropratio': 0.4,
 'dataset': 'training-datasets/iteration-0/UnaugmentedDataSet_Cloud_Deployment_SERMar27/Cloud_Deployment_SER_Dev95shuffle1.mat',
 'dataset_type': 'default',
 'decay_steps': 30000,
 'deterministic': False,
 'display_iters': 1000,
 'fg_fraction': 0.25,
 'global_scale': 0.4,
 'init_weights': ['/',
                  'home',
                  'marno',
                  'miniconda3',
               

✅ Type-repaired: /mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/dlc-models/iteration-0/Cloud_Deployment_SERMar27-trainset95shuffle1/train/pose_cfg.yaml
✅ Type-repaired: /mnt/Data/Projects/cloud_deployment/Scripts/Cloud_Deployment_SER-Dev-2026-03-27/dlc-models/iteration-0/Cloud_Deployment_SERMar27-trainset95shuffle1/test/pose_cfg.yaml

🔥 Starting training run. It's safe to let it chug now.
Selecting single-animal trainer

❌ Pipeline failed: unsupported operand type(s) for ** or pow(): 'str' and 'int'
